# 7회차 · 내 데이터로 모델 학습시키기(LoRA 샌드박스)

오늘은 모델의 **가중치를 직접 바꿈**. 내가 원하는 말투 · 형식을 정하고, 학습 데이터를 만들고, 설정을 바꿔 가며 여러 번 학습시켜 차이를 확인함.
9-1 에서 RAG 와 LoRA 가 각각 무엇을 바꾸고 무엇을 못 바꾸는지 실험으로 확인한 뒤, 학습한 Adapter 를 저장함. 수업 마지막에는 **최종 스튜디오 노트북**(`s07_final_studio.ipynb`)에서 LLM · 검색 · 그림 · 소리 부품의 코드를 직접 이어 붙여 나만의 챗봇과 API 를 조립함.

**전체 흐름부터.** 5회차와 6회차는 모델을 그대로 두고 입력을 바꿨음(지시문, 문서). 그 방법은 매번 지시문을 붙여야 하고 입력이 길어짐. 오늘은 반대로 **모델 쪽을 조금 고쳐서** 지시문 없이도 원하는 방식으로 답하게 만듦.

모델 전체(수십억 개의 숫자)를 다시 학습시키면 시간과 메모리가 너무 많이 듦. **LoRA** 는 원래 가중치(본체)는 얼려 두고, 옆에 작은 행렬(Adapter)만 붙여 그 부분만 학습하는 방법임. 두꺼운 교과서는 그대로 두고 얇은 보충 노트만 새로 쓰는 것과 비슷함. 그래서 학습 한 번이 몇 분이면 끝나고 결과 파일도 수십 MB 임.

`학습 목표(1단계)` → 학습 데이터 수백 건(3단계) → 학습(4단계) → 학습 과정 들여다보기(4-2~4-5) → 지시문 없이 물어 확인(5단계) → 설정을 바꿔 다시(6단계) → 문서 검색과 합침(9단계) → 체감 실험실(9-1) → 저장 · 미리 보기(10~12단계) → **최종 스튜디오 노트북**

RAG 와 LoRA 는 하는 일이 다름. **무엇을 아는가(지식)** 는 문서를 붙이는 RAG 가, **어떻게 답하는가(말투 · 형식)** 는 LoRA 가 잘 바꿈. 9단계에서 숫자로 확인함.

- 학습은 한 번에 2~4분임(학습 프로그램이 모델을 따로 올리는 1~3분 포함). 설정을 바꿔 **세 번 이상** 학습해 봄
- 끝낼 때 맨 아래 '결과 묶기'를 실행해 zip 파일을 내려받음

| 단계 | 하는 일 | 필수 |
|---|---|---|
| 0 | 시작 : 6회차 솔루션 불러오기 | 필수 |
| 1 | 학습 목표 정하기 | 필수 |
| 2 | 학습 전 답 확인 | 필수 |
| 3 | 학습 데이터 만들기 | 필수 |
| 4 | 첫 학습 | 필수 |
| 4-1~4-5 | LoRA 크기 계산, 학습 데이터가 들어가는 모습, Adapter 구성, 학습 과정 그림, 단계별 평가 | 필수(4-1은 권장) |
| 5 | 전후 비교 | 필수 |
| 6 | 설정 실험 : 다시 학습하고 곡선 비교 | 필수 |
| 7 | 일부러 문제 만들기 | 자율 시간 |
| 8 | 큰 GPU 써 보기(QLoRA · 큰 배치 · 다른 본체) | 자율 시간 |
| 9 | 네 구성 비교 | 필수 |
| 9-1 | 체감 실험실 : 같은 질문 네 구성, LoRA 로 지식 넣기, 학습 뒤 부작용 | 필수(지식 넣기는 3~5분) |
| 10 | 내 솔루션 완성 · 저장 | 필수 |
| 11 | 완성 모습 미리 보기 → 최종 스튜디오 노트북에서 직접 조립 | 필수 |
| 12 | 판단 기록 · 결과 묶기 | 필수 |
| 스튜디오 | `s07_final_studio.ipynb` : 부품 코드 조립 · API · 응용 · 추가 학습 | 필수(20:05~) |
| A~K | LoRA 크기 계산, 곡선 비교, 추가 학습 실험, 단계별 평가, 그림 모듈, 캐릭터 바꾸기(자율 시간) | 더 해 보기 |

7 · 8단계와 '더 해 보기 A~K'는 자율 시간에 함(수업 뒤에도 서버가 켜져 있는 동안 가능). 20:05에는 진행 단계와 상관없이 12단계까지 마치고 스튜디오 노트북으로 넘어감. 목표를 정하기 어려우면 맨 아래 '예시 과제'를 봄.

※ 강사가 11/22~23에 개인 서버에서 처음부터 끝까지 직접 돌려 본 노트북임. 숫자는 목표와 서버에 따라 조금씩 달라질수 있음

## 0. 시작
아래 두 셀을 차례로 실행함. 6회차에 저장한 솔루션(모델 · 문서 · 검색 설정)을 그대로 불러옴. 6회차 결과가 없으면 `lab.docs_panel()` 로 문서를 넣고 `lab.quiz_panel()` 로 확인 질문을 적은 뒤 이어 감.

In [ ]:
import sys, pathlib
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "lab.py").exists())
sys.path.insert(0, str(root / "src"))
import lab
lab.start(7)

In [ ]:
lab.load_solution()
lab.gpu()

## 1. 학습 목표 정하기
모델이 **어떻게 답하게 만들 것인지** 정함. 준비된 목표를 고르면 입력란이 채워지고, 그대로 쓰거나 고쳐 씀. 직접 새로 적어도 됨.
- **지시문** : 원하는 말투 · 형식 · 길이. 학습 데이터를 만들 때만 쓰고, 학습한 모델에는 주지 않음
- **표지 단어** : 목표대로 답했는지 판정하는 단어나 문장 끝 표현. 점수 계산에 씀
- **예시** : `질문 => 내가 원하는 답`. 내가 원하는 답의 모습 그대로 3건 이상 적음. 예시의 품질이 학습 결과를 좌우함

**왜 하나** : 학습은 "이런 질문에는 이렇게 답하라"는 본보기를 수백 번 보여 주는 일임. 본보기가 흐릿하면 결과도 흐릿함. 그리고 학습이 됐는지 숫자로 확인하려면 판정 기준이 필요한데, 표지 단어가 그 기준임(예 : 사극 말투라면 `하오`, `이오`). 표지 단어를 정할 수 없는 목표는 점수로 잴 수 없으므로 목표를 더 구체적으로 고침.

In [ ]:
lab.goal_panel()

## 2. 학습 전 답 확인
학습하기 전의 모델에게 지시문 없이 물어봄. 지금은 평범하게 답하는 것이 정상임. 같은 질문을 5단계에서 다시 물어 비교함.

**왜 하나** : 고치기 전의 모습을 봐 두지 않으면 학습으로 무엇이 바뀌었는지 말할 수 없음. 질문 두세 개를 정해 두고 답을 기억해 둠.

In [ ]:
lab.compare_panel()

## 3. 학습 데이터 만들기
학습에는 `질문 → 원하는 답` 쌍이 수백 건 필요함. 손으로 다 쓸 수 없으므로 지시문과 예시를 받은 **교사 모델**이 질문 수백 건에 목표대로 답하게 하고, 모델이 낸 답을 학습 데이터로 씀.

**왜 하나** : 지시문을 받으면 잘 답하는 모델(교사)의 답을 모아서, 지시문 없이도 그렇게 답하는 모델(학생)을 만드는 것임. 이때 교사와 학생은 같은 모델임. 지시문이 있을 때의 답을 지시문이 없을 때도 내도록 가르친다고 보면 됨.

| 바꿔 볼 것 | 값 | 볼 것 |
|---|---|---|
| 만들 건수 | 60 → 200 → 400 | 6단계에서 건수에 따라 말투 점수가 어떻게 달라지는가 |
| 내 문서 섞는 비율 | 0 → 0.4 → 0.7 | 9단계 '문서 + Adapter'의 말투 점수. 0 이면 문서를 붙였을 때 말투가 사라지기 쉬움 |
| 표지 단어 기준 | 켬 대 끔 | '기준에 못 미쳐 뺀 답'이 몇 건인가 |

- 진행 막대의 '한 번에 몇 건'이 GPU 메모리에 따라 정해짐. 메모리가 클수록 한꺼번에 많이 생성함
- 만들어진 표를 **꼭 읽어 봄**. 답이 목표와 다르면 1단계로 돌아가 지시문 · 예시를 고친 뒤 다시 만듦. 데이터가 틀리면 모델은 틀린 대로 배움

In [ ]:
lab.data_panel()

## 4. 첫 학습
기본 설정 그대로 '학습 시작'을 누름. Loss 곡선이 실시간으로 그려짐. 시작할 때 '학습 계획' 한 줄(학습 건수 ÷ 배치 = 에폭당 단계 × 에폭 = 전체 단계)이 먼저 나옴.

기본값으로 에폭이 끝날 때마다 Adapter 를 **중간 저장**함(곡선 아래 ▲). 4-5에서 이 중간 저장본으로 '학습이 진행될수록 얼마나 목표를 따라오는지' 확인함. 더 자세히 보려면 '중간 저장'을 6번 · 10번으로 고름.

**곡선 읽는 법** : **Loss** 는 모델의 답이 학습 데이터의 답과 얼마나 다른지 나타내는 값임. 내려가면 학습 데이터를 닮아 가고 있다는 뜻임. 실선은 학습에 쓴 데이터의 Loss, 점선은 학습에 쓰지 않고 따로 떼어 둔 검증 데이터의 Loss 임. 실선은 계속 내려가는데 점선이 다시 올라가기 시작하면, 모델이 데이터를 이해한 것이 아니라 외우고 있다는 신호임(과적합).

- 표에서 **학습한 파라미터 비율**과 **Adapter 크기**를 확인함. 전체의 1% 도 안되는 숫자만 고쳤는데 답이 바뀜
- 학습 중에 계기판을 보면 메모리가 모델만 올렸을 때보다 늘어나 있음. 학습은 계산 중간값을 저장해야 해서 메모리를 더 씀

![LoRA 기초 : 얼린 본체와 작은 행렬 두 개, rank에 따른 크기](img/lora_basics.png)

**LoRA 한 걸음 더**
- 모델 안의 큰 행렬 W(3584 × 3584)는 그대로 얼려 둠. 옆에 B(3584 × r)와 A(r × 3584) 두 개를 붙이고 **이 둘만** 학습함. 출력 = W·x + (α / r) · B·A·x
- rank 16, 28층의 q · k · v · o 에 붙이면 학습 파라미터는 10,092,544개(전체의 약 0.14%), Adapter 파일은 38.5MB 임. 11/22 실측과 숫자가 정확히 같음
- 크기는 rank 에 비례함. rank 를 4배로 올리면 파라미터도 파일도 4배. 붙일 층을 q · v 둘로 줄이면 절반
- Adapter 를 떼면 원래 모델로 돌아감. 본체 하나에 목표별 Adapter 를 여러 개 두고 바꿔 끼울수 있음. QLoRA 는 이 구조는 같고 얼려 둔 본체만 4bit 로 올림

※ 리허설 때 "0.14%만 바꿨는데 말투가 바뀌나?" 싶었는데 v1 하나로 말투 점수가 0 → 0.9 근처까지 올랐음. 아래 4-1 셀로 숫자를 직접 계산해 보면 감이 옴

In [ ]:
lab.train_panel()

**코드로 보기 : 학습 조절판이 실제로 실행하는 코드.** 조절판의 값은 아래 `LoraConfig`(무엇을 얼마나 붙일지)와 `SFTConfig`(어떻게 학습할지)로 들어가고, `SFTTrainer` 가 학습을 돌림. 학습 결과 표 위의 '실제로 실행한 명령'을 펼치면 조절판 값이 어떤 옵션으로 넘어갔는지 보임.

| 코드의 이름 | 조절판의 값 | 하는 일 |
|---|---|---|
| `r` · `lora_alpha` | rank(r), α(= 2 × r) | Adapter 행렬의 폭과 배율(α / r) |
| `target_modules` | LoRA 를 붙일 층 | q · k · v · o 등 어느 행렬 옆에 붙일지 |
| `learning_rate` · `num_train_epochs` · `per_device_train_batch_size` | 학습률 · 에폭 · 배치 | 한 번에 고치는 폭, 반복 횟수, 한 번에 보는 건수 |
| `lr_scheduler_type="cosine"` · `warmup_steps` | (고정) | 4-4 의 학습률 그림 모양 |
| `gradient_checkpointing` | (고정) | 메모리를 아끼려고 중간값을 버렸다가 다시 계산 |

In [ ]:
lab.code("lab_train.py", "lora = LoraConfig", 26)

### 4-1. LoRA 크기 직접 계산
본체 설정(층 수, 행렬 크기)을 읽어 rank 와 붙일 층에 따라 학습 파라미터가 몇 개인지, Adapter 가 몇 MB 인지 계산함. 4단계 결과 표의 숫자와 맞춰 봄.

In [ ]:
# 4-1. LoRA 크기 직접 계산 : 본체 설정을 읽어 rank별 학습 파라미터 수와 Adapter 크기를 계산함
from transformers import AutoConfig
mid = lab.cfg()["model"]                           # 6회차에 저장한 본체(기본 skt/A.X-4.0-Light)
c = AutoConfig.from_pretrained(mid)
h, L = c.hidden_size, c.num_hidden_layers
kv = c.num_key_value_heads * (getattr(c, "head_dim", None) or h // c.num_attention_heads)   # k · v 출력 크기

def lora_params(r, target="qkvo"):
    q = o = r * (h + h)                            # q · o : (h × h) 행렬 옆에 B(h × r) · A(r × h)
    k = v = r * (h + kv)                           # k · v : (h × kv) 행렬
    return L * (q + k + v + o if target == "qkvo" else q + v)

print(f"본체 {mid} · hidden {h} · 층 {L} · k·v 출력 {kv}")
for r, tg in [(2, "qkvo"), (8, "qkvo"), (16, "qkvo"), (64, "qkvo"), (16, "qv")]:
    n = lora_params(r, tg)
    print(f"rank {r:>2} · {tg:<4} → 학습 파라미터 {n:>11,}개 · Adapter 약 {n * 4 / 2**20:6.1f}MB")
print("기본 설정(rank 16 · qkvo)의 11/22 실측 : 10,092,544개 · 38.5MB")

### 4-2. 학습 데이터가 모델에 들어가는 모습
학습 데이터 한 건에 채팅 템플릿을 씌운 실제 글자를 봄. **회색(질문 부분)은 Loss 계산에서 빠지고, 초록(답 부분)만 맞히도록 학습함.** 숫자를 바꿔 다른 건도 봄(예 : `lab.data_view(5)`). 문서가 붙은 예시는 질문 부분이 길게 나옴.

**왜 하나** : '학습한다'는 말이 실제로는 무엇인지 확인하는 단계임. 모델은 답 부분의 토큰을 하나씩 맞히는 연습을 함. 그래서 질문을 아무리 많이 넣어도 답이 흐릿하면 배울 것이 없음.

In [ ]:
lab.data_view(0)

### 4-3. 내 Adapter 구성 들여다보기
방금 학습한 Adapter 파일을 열어 **어느 층의 어느 행렬에 몇 개의 숫자가 붙었는지** 표로 보고, 학습으로 각 행렬이 **얼마나 바뀌었는지(‖ΔW‖)** 를 층 × 행렬 그림으로 봄. 이름을 넣으면 그 Adapter 를 봄(예 : `lab.lora_view("v2")`).

- 표의 합계가 4단계 결과 표의 '학습한 파라미터'와 같은지 확인함
- 그림에서 밝을수록 많이 바뀐 행렬임. 같은 rank 라도 층마다 바뀐 크기가 다름. 어느 층 · 어느 행렬이 가장 많이 바뀌었는지 적어 둠

※ 11/23 리허설(사극 말투)에서는 두 번 모두 o · q 가 k · v 보다 두세 배 많이 바뀌었음. 가장 밝은 행렬은 학습마다 달랐음(데이터 60건 17층 o, 162건 27층 o). 목표마다 달라지니 내 그림과 비교해 봄

In [ ]:
lab.lora_view()

### 4-4. 학습 과정 자세히 보기
학습 한 번의 전 과정을 그림 네 개로 봄. 점선은 에폭 경계, ▲는 중간 저장 지점임.

| 그림 | 보는 것 | 정상인 모습 |
|---|---|---|
| ① Loss | 학습 데이터의 답과 얼마나 다른가 | 처음에 빠르게 내려가고 점점 평평해짐. 검증 Loss 도 함께 내려감 |
| ② 토큰 정확도 | 답의 다음 토큰을 맞힌 비율 | Loss 와 반대로 올라감. 에폭이 바뀌는 지점에서 한 번 더 뜀(같은 데이터를 다시 봄) |
| ③ 학습률 | 한 번에 고치는 폭 | 처음 몇 단계 동안 올라갔다가(워밍업) 코사인 모양으로 줄어듦 |
| ④ 기울기 크기 | 한 번에 고치려는 정도 | 처음에 크고 점점 작아짐. 갑자기 튀면 학습률이 너무 큰 것 |

표의 '에폭당 단계'는 학습 건수 ÷ 배치(올림)임. 배치를 키우면 단계 수가 줄고, 에폭을 늘리면 단계 수가 늘어남.

In [ ]:
lab.train_report()

### 4-5. 단계별로 얼마나 따라오나
중간 저장본마다 **검증 질문(학습에 쓰지 않은 질문)** 으로 말투 점수를 재서, 학습 단계가 지날수록 목표를 얼마나 따라오는지 그림으로 봄. 같은 질문 하나에 대한 단계별 답도 나란히 나오므로, 말투가 언제부터 바뀌기 시작하는지 눈으로 확인함. 1~2분 걸림.

- 왼쪽 그림 : 말투 점수(실선)와 검증 Loss(점선). 말투 점수가 첫 에폭에 거의 다 오르고 그 뒤로 평평하면 에폭을 줄여도 됨
- 오른쪽 그림 : 학습으로 바뀐 크기 ‖ΔW‖. 학습 전은 0 이고 단계가 지날수록 커짐. 커지는데 점수가 그대로면 더 학습해도 얻는 것이 적음
- 질문을 바꾸려면 `lab.progress_eval(q="내 질문")`

In [ ]:
lab.progress_eval()

## 5. 전후 비교
2단계와 같은 질문, 그리고 학습 데이터에 없던 새 질문으로 물어봄. 표지 단어는 색으로 표시됨. 아래 두 번째 셀은 검증 질문 20건으로 **말투 점수**를 재서 막대로 그림.

**왜 하나** : 학습 데이터에 있던 질문에 잘 답하는 것은 당연함. **처음 보는 질문**에도 목표대로 답해야 학습이 된 것임. 그래서 말투 점수는 학습에 쓰지 않은 질문으로 잼.

In [ ]:
lab.compare_panel()

In [ ]:
lab.style_eval()

**코드로 보기 : Adapter 를 켜고 끄는 코드.** 전후 비교 조절판은 아래처럼 같은 본체에 Adapter 를 켠 채로 한 번, `disable_adapter()` 안에서 한 번 생성함. 본체는 하나이고 Adapter 만 켜고 끄는 것임. `print(model)` 의 일부를 보면 원래 `Linear` 옆에 `lora_A` · `lora_B` 가 붙어 있음.

In [ ]:
# 5단계 코드 : 같은 본체로 Adapter 를 켜고 끄며 생성(질문과 Adapter 이름을 바꿔 봄)
import torch
AD = lab.runs(show=False)[-1]["이름"]                       # 가장 최근 Adapter
tok, model, _ = lab.llm_parts(AD)                           # AD 를 켠 모델
msgs = [{"role": "user", "content": "비 오는 날 점심 메뉴는?"}]
ids = tok(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True), return_tensors="pt").to(model.device)
with torch.no_grad():
    on = model.generate(**ids, max_new_tokens=80, do_sample=False, pad_token_id=tok.pad_token_id)
    with model.disable_adapter():                           # Adapter 를 잠시 끔
        off = model.generate(**ids, max_new_tokens=80, do_sample=False, pad_token_id=tok.pad_token_id)
n = ids["input_ids"].shape[1]
print("[Adapter 켬]", tok.decode(on[0, n:], skip_special_tokens=True))
print("[Adapter 끔]", tok.decode(off[0, n:], skip_special_tokens=True))
print("\n[모델 구조 일부]")
print(model.base_model.model.model.layers[0].self_attn.q_proj)

## 6. 설정 실험
4단계 조절판으로 돌아가 **설정 하나만** 바꿔 다시 학습함(이름은 비워 두면 v2, v3 … 로 붙음). 실행 전에 결과를 예측해 봄.

**설정이 뜻하는 것**
- **rank(r)** : Adapter 행렬의 크기. 크면 더 많은 것을 배울 수 있지만 파일이 커지고, 데이터가 적을 때는 외우기 쉬워짐
- **학습률** : 한 번에 얼마나 크게 고칠지. 너무 작으면 거의 안 변하고, 너무 크면 Loss 가 내려가지 않고 튐
- **에폭** : 같은 데이터를 몇 번 반복해 볼지. 많을수록 Loss 는 내려가지만 과적합 위험이 커짐
- **배치 크기** : 한 번에 몇 건을 묶어 계산할지. 크면 빠르지만 메모리를 더 씀

| 바꿔 볼 것 | 살펴볼 것 |
|---|---|
| rank 2 대 64 | 학습한 파라미터 수 · Adapter 크기 · 말투 점수 |
| 에폭 1 대 6 | Loss 가 어디까지 내려가는가 · 답이 학습 데이터를 그대로 외우는가 |
| 학습 건수 40 대 전부 | 데이터가 적어도 되는가 |
| 붙일 층 q · v 대 전체 | 학습 시간 · 점수 |

두 번 이상 더 학습한 뒤 아래 셀로 Loss 곡선을 한 그림에 함께 그리고 점수를 다시 잼.

In [ ]:
lab.loss_plot()
lab.style_eval()

## 7. 일부러 문제 만들기(자율 시간)
잘못된 설정이 어떤 모습으로 나타나는지 직접 봄. 조절판에서 아래처럼 고르고 학습한 뒤 5단계 조절판으로 답을 읽어 봄.

**왜 하나** : 실제로 학습을 돌리다 보면 결과가 이상할 때가 훨씬 많음. 그때 곡선과 답의 모습만 보고 원인을 짐작할수 있어야 함. 문제를 일부러 만들어서 그 모습을 미리 봐 두는 것임.

- **학습률 0.01** : Loss 가 내려가지 않거나 치솟음. 답이 깨지는지 확인함
- **에폭 12 · 학습 건수 20** : Loss 는 0 에 가까워지는데 새 질문에는 맞지 않는 답을 함(과적합)
- 1단계에서 예시를 일부러 틀리게 적고 데이터를 다시 만들면, 모델은 틀린 답을 그대로 배움

일부러 문제를 만든 학습은 최종 솔루션으로 고르지 않음. 이름을 `bad1` 처럼 붙여 두면 헷갈리지 않음.

## 8. 큰 GPU 써 보기(자율 시간)
- **QLoRA** : 본체 자료형을 4bit 로 바꿔 학습함. 표의 '학습 메모리'와 걸린 시간, 말투 점수를 bf16 과 비교함. 본체를 4bit 로 줄여 올리면 메모리는 크게 줄고 속도는 조금 느려짐. 작은 GPU 에서 학습할 때 쓰는 방법임
- **큰 배치** : 배치를 32 · 64 로 키움. 한 번에 많이 보지만 갱신 횟수가 줄어듦. 점수가 어떻게 되는지 봄
- **다른 본체** : '학습할 본체'를 `Qwen3-8B` 로 바꿔 같은 데이터로 학습함. 학습한 Adapter 는 그 본체에서만 쓸 수 있으므로, 아래 셀로 본체를 올리고 비교함

**왜 하나** : 같은 학습이 GPU 크기에 따라 어떻게 달라지는지 보는 단계임. 96GB 에서는 원래 정밀도(bf16)로 큰 배치를 돌릴 수 있고, 24GB 에서는 4bit 와 작은 배치를 써야 함. 나중에 내 GPU 로 학습할 때 무엇을 줄여야 하는지 미리 알아 둠.

In [ ]:
lab.gpu()
# 다른 본체로 학습했으면: lab.load('Qwen/Qwen3-8B') 뒤 lab.compare('질문') 으로 비교하고, 끝나면 lab.use('A.X-4.0-Light') 로 돌아옴

## 9. 네 구성 비교
6회차에 적은 확인 질문 전체를 네 가지 구성으로 풀어 **내용 점수**(기대 단어)와 **말투 점수**(표지 단어)를 함께 봄.
- 학습 전 · 문서 붙임(RAG) · Adapter(LoRA) · 문서 + Adapter

**왜 하나** : 오늘 수업의 결론이 이 그림 한 장임. 내용 점수는 문서를 붙일 때 오르고, 말투 점수는 Adapter 를 쓸 때 오름. 둘을 같이 쓰면 둘 다 오르는지, 아니면 문서를 붙이는 순간 말투가 사라지는지 내 데이터로 확인함. 사라졌다면 3단계의 '내 문서 섞는 비율'을 올려 데이터를 다시 만들고 학습함.

In [ ]:
lab.four()

## 9-1. 체감 실험실 : RAG 와 LoRA 가 바꾸는 것과 못 바꾸는 것
9단계의 숫자를 실제 답으로 확인함. 세 실험을 차례로 함.

**① 같은 질문, 네 구성** : 질문 하나를 네 구성으로 물어 답을 나란히 봄. 노랑은 기대 단어(내용), 초록은 표지 단어(말투)임. 내 확인 질문 가운데 하나를 넣으면 기대 단어까지 칠해짐.

**② LoRA 로 지식을 넣으면?** : 별빛 동아리의 '2027년 변경 사항' 네 가지(회비 3만 5천 원, 셋째 주 토요일 모임, 과학관 204호, 망원경 5일)를 질문 12개로 학습한 Adapter(`fact`)와, 같은 내용을 문서로 붙인 경우를 비교함. 질문을 학습한 그대로 · 바꿔서 · 둘을 엮어서 · 관련 없이 물음. 모델 올리기를 포함해 3~5분 걸리므로 먼저 실행해 두고 기다리는 동안 ① · ③을 해도 됨.

| 볼 것 | 11/23 강사 서버 결과(EPOCHS 15) | 뜻 |
|---|---|---|
| 그대로 물음 | LoRA · 문서 붙임 모두 맞힘 | 학습한 문장은 외움 |
| 바꿔 물음 · 엮어 묻기 | 회비 · 모임 · 망원경은 LoRA 도 맞힘. '과학관 몇 층'에는 LoRA 가 '두 층에 있어요'라고 지어냄, 문서 붙임은 '204호, 층수는 문서에 없음' | 외운 사실 밖의 질문에는 지어냄. 문서는 근거와 모름을 함께 줌 |
| 관련 없는 질문 | 라면 질문에 '2027년부터 별빛 관측소는 …', 파이썬 질문에 '20228년부터 파이썬에서는 …'으로 답함 | 학습 문장의 모양이 다른 질문으로 번짐 |
| 같은 실험을 EPOCHS 5 로 | 네 사실을 하나도 못 맞힘('5만 5천 원', '22동 411호'처럼 숫자를 지어냄), 번짐은 없음 | 덜 배우면 문장 모양만 배우고 사실은 지어냄 |

**③ 학습 뒤 부작용** : 목표와 상관없는 질문(영어 · 계산 · 코드 · JSON · 요약)에 학습 전 모델과 내 Adapter 가 어떻게 답하는지 봄. 말투가 번지거나 형식이 깨지는지 관찰 열을 확인함.

※ 11/23 밤 강사 서버(A100)에서 EPOCHS 5 와 15 로 각각 돌려 본 결과가 위 표임. 15 는 마지막 Loss 가 0.0006 까지 내려가 학습 문장을 거의 그대로 외웠음. 결과는 학습할 때마다 조금씩 다름
※ 문서 붙임은 라면 질문에 '정보를 제공할 수 없습니다'라고 답했음. 6회차의 '문서에 없으면 모른다고 답함' 규칙 때문이며, 문서 밖의 일반 질문까지 막는 RAG 의 한계임

In [ ]:
lab.four_view("동아리 회비는 얼마고 언제 내?")     # 내 확인 질문으로 바꿈

In [ ]:
# ② LoRA 로 지식 넣기 : 값을 바꿔 다시 해 봄(EPOCHS 를 5 로 줄이면 무엇을 배우는가, 번짐은 줄어드는가)
EPOCHS = 15
REPEAT = 4          # 학습 데이터를 몇 번 되풀이해 넣을지(질문 12개 × REPEAT)
fact = lab.knowledge_lab(epochs=EPOCHS, repeat=REPEAT)

In [ ]:
lab.side_effects()      # 저장한 솔루션의 Adapter(저장 전이면 가장 최근에 학습한 Adapter, fact 는 뺌)로 확인. 이름을 넣어도 됨

<details><summary>확인 질문 : 동아리 회비가 내년에 또 바뀐다. LoRA 와 RAG 가운데 무엇으로 반영하는가, 그 이유는</summary>

문서(RAG)로 반영함. 문서 한 줄만 고치면 바로 맞고, 근거도 보여 줄 수 있음. LoRA 는 다시 학습해야 하고, ②에서 본 것처럼 바꿔 물으면 틀리며 다른 질문에 번짐. LoRA 는 말투 · 형식 · 판단 규칙처럼 자주 바뀌지 않는 '방식'을 가르칠 때 씀
</details>

## 10. 내 솔루션 완성
가장 마음에 드는 Adapter 이름을 넣어 저장함. 저장한 Adapter 가 11단계 챗봇의 기본값이 되고, 8회차부터는 저장한 솔루션을 불러 써서 서비스로 만듦.

In [ ]:
lab.runs()

In [ ]:
lab.save_solution(adapter='v1')     # 고른 Adapter 이름으로 바꿈

## 11. 완성 모습 미리 보기 → 최종 스튜디오에서 직접 조립
아래는 수업 마지막에 **최종 스튜디오 노트북**(`s07_final_studio.ipynb`)에서 부품부터 직접 조립할 챗봇의 완성 모습임. 이 셀에서는 lab 이 미리 이어 둔 것을 한 번만 돌려 보고, 조립은 스튜디오에서 코드로 함(소리 받아쓰기 · 읽어 주기 · 웹 화면 · API 포함). 질문 하나에 아래 네 가지가 차례로 일어남.

![최종 챗봇의 흐름 : 검색 → Adapter 로 답 → 상태 정리 → 그림](img/final_bot_flow.png)

| 차례 | 하는 일 | 쓰는 것 | 배운 회차 |
|---|---|---|---|
| ① 검색 | 내 문서에서 질문과 가까운 청크를 찾음 | 임베딩 · 하이브리드 검색(RAG) | 6회차 |
| ② 답 | 찾은 청크를 붙여, 학습한 Adapter 를 얹은 모델이 답함 | 본체 + LoRA Adapter | 7회차 |
| ③ 상태 정리 | 같은 모델이 질문과 답을 읽고 지금 분위기(기쁨 · 차분 · 걱정 · 놀람 · 진지 · 장난)와 장면, 그림 묘사를 JSON 으로 씀 | 지시문 · 형식 지정 | 5회차 |
| ④ 그림 | 상태에 맞는 캐릭터 그림을 그림 | 그림 생성 모델(sd-turbo, 1단계) | 오늘 처음 |

**왜 하나** : 실제 서비스의 챗봇은 모델 하나로 끝나지 않고 여러 부품을 이어 붙여 만듦. 지식은 RAG 가, 말투는 LoRA 가, 화면 연출은 형식을 지정한 출력(JSON)이 맡고, 그 결과를 다른 모델(그림)이 받아 씀. 부품마다 걸린 시간이 답 아래 막대로 나오므로 어디가 느린지도 봄.

- 처음 그림을 그릴 때 그림 모듈 설치와 모델 내려받기로 1~2분 걸림. 그 다음부터는 한 장에 1초 안팎
- 같은 분위기는 같은 seed 로 그려 캐릭터가 비슷하게 유지됨. 캐릭터 설명을 바꾸면 그림 속 인물이 바뀜
- 그림 모델(sd-turbo)의 사용 조건은 모델 카드의 라이선스를 따름. 수업에서는 비상업 교육용으로만 쓰고, 수업 밖에서 쓸 때는 라이선스를 확인함. 그림 모델에는 안전 필터가 없어 lab 이 묘사에 걸리는 낱말이 있으면 다른 그림으로 바꿈

**한 턴 미리 보기.** 답 · 근거 · 말투 점수 · 분위기 · 그림 · 부품별 시간이 한 카드에 나옴. 질문을 내 문서 주제로 바꿈. 처음 그림을 그릴 때는 설치와 내려받기로 1~2분 걸림.

**다음 할 일** : 12단계(판단 기록 · 결과 묶기)를 마치고 이 노트북의 커널을 끈 뒤(Kernel → Shut Down Kernel) `s07_final_studio.ipynb` 를 엶.

In [ ]:
r = lab.bot("동아리 회비는 얼마고 언제 내?")       # 내 문서에 답이 있는 질문으로 바꿈
lab.code("bot")                                     # 이 한 줄 뒤에서 일어나는 일(스튜디오에서 직접 조립함)

## 12. 판단 기록 · 결과 묶기
어떤 설정의 Adapter 를 골랐고 왜 골랐는지, 무엇이 기대와 달랐는지 적음. 근거가 되는 숫자를 두 개 이상 넣음(말투 점수, 학습 시간, Adapter 크기 등).
끝으로 결과 묶기를 실행하고 zip 파일을 내려받음.

In [ ]:
lab.note_panel()

In [ ]:
lab.pack()

---
## 더 해 보기 A~K(자율 시간)
필수 단계를 마쳤으면 아래에서 골라 해 봄. 셀 위쪽 몇 줄(질문, 이름)만 내 것으로 고치면 됨. D · E · I 는 학습을 한 번 더 돌리므로 1~5분씩 걸림. 워크북 '더 해 보기' 절에 같은 목록과 적을 것이 정리돼 있음.

| 번호 | 할 일 | 걸리는 시간 | 볼 것 |
|---|---|---|---|
| A | 내 Adapter 크기를 계산값과 맞춰 보기 | 3분 | 표의 MB 와 계산이 맞는가 |
| B | Adapter 표와 Loss 곡선 한 번에 보기 | 3분 | 어느 학습이 덜 내려갔는가 |
| C | 처음 보는 질문 다섯 개로 전후 비교 | 5분 | 학습 데이터에 없던 질문에도 말투가 나오는가 |
| D | 학습 건수 50건으로 다시 학습 | 5분 | 데이터가 적으면 말투 점수가 어떻게 되는가 |
| E | rank 4로 다시 학습 | 5분 | 파라미터가 1/4이면 점수는 1/4인가 |
| F | 지시문으로 시킨 답 대 Adapter 답 | 3분 | 지시문 없이도 같은 말투가 나오는가 |
| G | 학습 데이터 들여다보기 | 3분 | 모델이 무엇을 보고 배웠는가 |
| H | 스스로 점검 일곱 문제 | 5분 | 접힌 답과 내 답 비교 |
| I | 중간 저장 10번으로 다시 학습하고 단계별 평가 | 7분 | 말투가 몇 단계째부터 바뀌는가, 에폭을 줄여도 되는가 |
| J | 그림 모듈만 따로 : 단계 수와 seed | 5분 | 1단계와 4단계의 차이, 같은 seed 는 같은 그림인가 |
| K | 캐릭터 · 부품을 바꿔 최종 챗봇 | 7분 | 캐릭터 설명 · Adapter · 검색을 바꾸면 답과 그림이 어떻게 달라지는가 |

※ D · E · I 는 GPU 32GB 서버에서 3~5분씩 걸림(올려 둔 모델을 잠시 내렸다가 다시 올림). 기다리는 동안 G 나 J 를 해도 됨

### A. 내 Adapter 크기를 계산값과 맞춰 보기
4-1 셀의 계산식으로 내 Adapter 의 크기를 예상하고, 실제 파일 크기와 비교함. 4-1 셀을 먼저 실행함.

In [ ]:
# A. 내 Adapter : 계산값 대 실제 크기(4-1 셀의 lora_params 를 씀)
for row in lab.runs(show=False):
    info = lab._run_info(row["이름"])
    n = lora_params(row["r"], info.get("target", "qkvo") if info.get("target") in ("qkvo", "qv") else "qkvo")
    print(f"{row['이름']:<8} rank {row['r']:>3} · 붙일 층 {info.get('target', '?'):<4} · 계산 {n * 4 / 2**20:6.1f}MB · 실제 {row['Adapter(MB)']}MB · 말투 점수 {row['말투 점수']}")

### B. Adapter 표와 Loss 곡선 한 번에 보기
지금까지 학습한 Adapter 를 표로 보고, Loss 곡선을 한 그림에 함께 그림. 이름을 넣으면 그 학습만 그림(예 : `lab.loss_plot("v1", "v3")`).

In [ ]:
# B. 학습 기록 표와 Loss 곡선
lab.runs()
lab.loss_plot()

### C. 처음 보는 질문 다섯 개로 전후 비교
학습 데이터에 없을 법한 질문을 다섯 개 넣어 학습 전 모델과 최근 Adapter 들의 답을 나란히 봄. 표지 단어는 색으로 표시됨.

In [ ]:
# C. 처음 보는 질문으로 전후 비교(내 목표에 맞게 질문을 바꿈)
qs = ["주말에 혼자 할 만한 취미를 추천해 줘", "비 오는 날 점심 메뉴는?", "시험 공부가 잘 안될 때는 어떻게 해?",
      "동아리 신입 회원에게 한마디 해 줘", "오늘 날씨가 추운데 옷은 뭘 입을까?"]
for q in qs:
    lab.compare(q)

### D. 학습 건수 50건으로 다시 학습
학습 데이터 가운데 50건만 써서 학습함. 끝나면 말투 점수를 기본 학습과 비교함. 예측을 먼저 적어 둠.

In [ ]:
# D. 학습 건수 50건(1~4분 걸림)
lab.train(run="n50", n=50)
lab.style_eval()

### E. rank 4로 다시 학습
rank 를 16에서 4로 줄이면 학습 파라미터와 Adapter 크기는 1/4 이 됨. 말투 점수도 1/4 이 되는지 봄.

In [ ]:
# E. rank 4(1~4분 걸림)
lab.train(run="r4", r=4)
lab.style_eval()

### F. 지시문으로 시킨 답 대 Adapter 답
학습 전 모델에 목표 지시문을 붙여 물은 답과, 지시문 없이 Adapter 로 답한 결과를 비교함. 프롬프트로도 말투는 바뀌지만 매번 지시문을 붙여야 함.

In [ ]:
# F. 지시문 대 Adapter(질문을 바꿔 봄)
q = "주말에 뭐 하면 좋을까?"
g = lab.goal()
print("[학습 전 모델 + 지시문]")
print(lab.llm(q, system=g.get("instruction")))
print()
lab.compare(q)                                     # 지시문 없이 학습 전 · Adapter 답을 나란히 봄

### G. 학습 데이터 들여다보기
모델이 무엇을 보고 배웠는지 학습 데이터 몇 건을 직접 읽어 봄. 질문 · 답의 길이와 문서가 붙은 비율도 봄.

In [ ]:
# G. 학습 데이터(my/train.jsonl) 들여다보기
import json
rows = [json.loads(x) for x in open(lab.MY / "train.jsonl", encoding="utf-8")]
with_doc = sum(1 for r in rows if len(r["prompt"]) > 1)
print(f"학습 데이터 {len(rows)}건 · 문서가 붙은 것 {with_doc}건 · 답 평균 {sum(len(r['completion'][0]['content']) for r in rows) / len(rows):.0f}자")
for r in rows[:5]:
    print("-", r["q"][:40], "→", r["completion"][0]["content"][:80].replace("\n", " "))

### H. 스스로 점검
답을 먼저 한두 줄로 적어 보고 펼쳐서 비교함.

<details><summary>1. Adapter 를 떼면 모델은 어떻게 되는가. 학습 중에 본체가 바뀌었는가</summary>

원래 모델로 돌아감. 본체 가중치는 얼려 두었으므로 바뀌지 않았음. 그래서 본체 하나에 Adapter 여러 개를 바꿔 끼울 수 있음
</details>

<details><summary>2. rank 16 → 64로 올리면 학습 파라미터와 Adapter 크기는 몇 배가 되는가</summary>

둘 다 4배. 10,092,544개 · 38.5MB → 40,370,176개 · 154MB. 점수는 4배가 되지 않음
</details>

<details><summary>3. 학습 Loss 는 계속 내려가는데 검증 Loss 가 오르기 시작했다. 무엇을 의심하는가</summary>

과적합(학습 데이터를 외움). 에폭을 줄이거나, 학습 건수를 늘리거나, 학습률을 낮춤
</details>

<details><summary>4. 새로 바뀐 동아리 회비(지식)는 LoRA 로 가르치는 것이 좋은가</summary>

문서(RAG)로 넣는 것이 맞음. 지식은 자주 바뀌고 근거를 보여 줘야 함. LoRA 는 말투 · 형식 · 판단 규칙에 맞음(9단계 네 구성 비교)
</details>

<details><summary>5. 학습 데이터 답의 30%가 틀렸다면 모델은 무엇을 배우는가</summary>

틀린 답의 형식과 내용도 함께 배움. 데이터가 결과를 정함. 7단계와 발표 자료의 '학습 정답 30%가 틀렸을 때' 장을 봄
</details>

<details><summary>6. 4-5 그림에서 말투 점수가 첫 에폭에 0.9까지 오르고 그 뒤로 그대로다. 무엇을 바꿔 볼 수 있는가</summary>

에폭을 1~2로 줄여도 됨. 학습 시간이 줄고 과적합 위험도 줄어듦. 대신 검증 Loss 와 새 질문의 답을 함께 확인함
</details>

<details><summary>7. 11단계 챗봇에서 내 문서의 새 회비 금액이 틀리게 나온다. RAG · LoRA · 그림 가운데 어디를 먼저 보는가</summary>

RAG(검색). 근거 줄에 회비 문서가 나왔는지 봄. 지식은 검색과 문서가, 말투는 Adapter 가, 분위기와 그림은 상태 정리와 그림 모델이 맡음. 틀린 부분에 맞는 부품을 고침
</details>

### I. 중간 저장 10번으로 다시 학습하고 단계별 평가
같은 데이터로 에폭 4, 중간 저장 10번으로 학습한 뒤 학습 과정 그림과 단계별 평가를 봄. 4-5보다 촘촘하게 보이므로 말투가 바뀌기 시작하는 지점을 찾기 쉬움.

In [ ]:
# I. 촘촘한 중간 저장(학습 2~5분 + 평가 1~2분)
lab.train(run="fine", epochs=4, snaps=10)
lab.train_report("fine")
lab.progress_eval("fine")

### J. 그림 모듈만 따로 : 단계 수와 seed
같은 묘사와 seed 로 단계 수만 1 · 2 · 4로 바꿔 그림. 이어서 seed 만 바꿔 그림. 묘사는 영어로 씀.

In [ ]:
# J. 단계 수와 seed(묘사를 바꿔 봄)
p = "a cheerful scholar in white hanbok holding a fan, autumn palace garden, storybook illustration"
for st in (1, 2, 4):
    lab.image(p, seed=7, steps=st)
for sd in (8, 9):
    lab.image(p, seed=sd)

### K. 캐릭터 · 부품을 바꿔 최종 챗봇
같은 질문을 캐릭터 설명, Adapter, 검색을 바꿔 가며 물어봄. 부품 하나만 바꿨을 때 무엇이 달라지는지 봄.

In [ ]:
# K. 캐릭터 · 부품 바꾸기(질문과 캐릭터를 내 것으로 바꿈)
q = "시험이 내일인데 하나도 못 했어"
lab.bot(q)                                                     # 지금 설정
lab.bot(q, character="고양이 탐정 : 탐정 모자를 쓰고 돋보기를 든 회색 고양이")   # 캐릭터만 바꿈
lab.bot(q, adapter=None)                                       # Adapter 를 뗌(말투 비교)
lab.bot(q, rag=False, draw=False)                              # 검색 · 그림을 끔(부품별 시간 비교)

---
### 예시 과제(목표를 정하지 못했을 때 하나를 고름)
1. **말투 챗봇** : 준비된 목표(사극 말투, 해적 선장)를 고르거나 좋아하는 캐릭터의 말투를 직접 적음. 지시문 없이 말투 점수 0.8 이상을 목표로 함
2. **세 줄 요약 모델** : 어떤 질문에도 정확히 세 줄로 답하게 함. 표지 단어 대신 형식이 지켜졌는지를 5단계 답에서 직접 확인함
3. **우리 동아리 상담원** : 6회차 문서 챗봇에 친절한 상담원 말투를 입힘. 9단계에서 '문서 + Adapter'의 내용 점수와 말투 점수가 둘 다 0.8 이상이 되게 함

### 도전 과제
- **가장 작은 Adapter** : 말투 점수 0.8 을 넘는 가장 작은 rank 와 가장 적은 학습 건수를 찾음
- **30초 학습** : 에폭 · 건수 · 배치를 조정해 학습 시간을 30초 안으로 줄이고도 점수를 유지함
- **말투 두 개** : 서로 다른 목표로 Adapter 를 두 개 학습하고 대화 창의 Adapter 선택 상자로 바꿔 가며 써 봄. 본체는 하나인데 Adapter 만 바꿔 끼우는 것임
- **문서를 붙여도 말투 유지** : '내 문서 섞는 비율' 0 과 0.6 으로 각각 학습해 9단계 결과를 비교함
- **캐릭터 챗봇 완성** : 목표 말투에 어울리는 캐릭터 설명을 정하고, 분위기가 다른 질문 여섯 개로 여섯 가지 분위기 그림을 모두 뽑아 봄
- **가장 빠른 챗봇** : 11단계 답 아래 시간 막대를 보고 가장 오래 걸리는 부품을 줄임(답 길이 · 붙일 청크 수 · 그림 단계 수)
- 조절판 없이 직접 부르기 : `lab.train(r=8, epochs=2, snaps=6)`, `lab.progress_eval()`, `lab.lora_view()`, `lab.bot("질문")`, `lab.four()`